# E2SFCA Analysis for EmOC Accessibility
## Kano State, Nigeria

This notebook implements the Enhanced Two-Step Floating Catchment Area (E2SFCA) method to measure spatial accessibility to Emergency Obstetric Care (EmOC) facilities.

**Prerequisites**: Complete Notebook 01 (Data Preparation) first

This notebook:
1. Loads processed data from Notebook 01
2. Calculates distance matrices between demand and supply
3. Applies E2SFCA method for each facility category
4. Calculates accessibility scores
5. Classifies areas by deprivation level
6. Saves results for analysis in Notebook 03

**Method**: E2SFCA (Luo & Qi, 2009)[]

> Note: This notebook requires the [environment dependencies](requirements.txt) to be installed
> as well as either an [openrouteservice API key](https://openrouteservice.org/dev/#/signup) or a local instance of the ORS server.

## Workflow Summary:

The notebook gives an overview of the distribution of centres offering EmOC in the city, their classification and how they can be accessed during an emergency. Open source data from OpenStreetMap and tools (such as the openrouteservice) were used to create accessibility measures. Spatial analysis and other data analytics functions led to generating outputs within the 100x100m grid cells that categorised them into three levels: low, medium, and high.

* **Preprocessing**: Get data for EmOC facilities.
* **Analysis for Offer**:
    * Filter or classify EmOC facilities based on discussed criteria.
    * Visualise EmOC faccilities in their categories.
* **Analysis for Accessibility**:
    * Compute travel times to facilities using openrouteservice API or other routing services.
    * Generate areas for low, medium and high categories based on discussed criteria.
* **Analysis for Demmand**:
    * Downscale the popluation data to the 100x100m grid cells.
    * Derive socio-economic descriptors based on discussed criteria.

* **Result**: Generate results as GIS-compatible files.


### Datasets and Tools:
* [openrouteservice](https://openrouteservice.org/) - generate OD-Matrix on the OpenStreetMap road network

#  Workflow

Make sure you have the required packages installed. You can install them using pip:

```bash
pip install -r requirements.txt
```

This study integrates various Python geospatial analysis libraries and packages to support spatial data processing, visualization, and isochrone generation. The os module is used to interact with the operating system, managing file paths and reading environment variables such as API keys. folium library along with its MarkerCluster plugin, facilitates the creation of interactive maps for visualizing large-scale geospatial data. The openrouteservice.client serves as an interface to the OpenRouteService API, enabling the extraction of isochrones. pandas library for data analysis, provides functions for analyzing, cleaning, exploring, and manipulating data, while fiona supports reading and writing real-world data using multi-layered GIS formats, such as shapefiles. The shapely package is employed for the manipulation and analysis of planar geometric objects.

## Setting up the virtual environment

```bash
# Create a new virtual environment
python -m venv .venv
activate .venv/bin/activate
pip install -r requirements.txt
```

## To run your notebook in VS Code

```bash
pip install -U ipykernel
python -m ipykernel install --user --name=.venv
```

In [1]:
import geopandas as gpd
import os
import numpy as np
import pandas as pd

import openrouteservice
from dotenv import load_dotenv

import rasterio
from rasterio.mask import mask

from shapely.geometry import Point
from pathlib import Path
from shapely.geometry import Polygon

import requests
import math
from math import *
from sklearn.preprocessing import MinMaxScaler

### Setting up the public API Key from OpenRouteService
In this study, users must obtain an ORS Matrix API key from the [OpenRouteService](https://openrouteservice.org/) platform and subsequently interacted with the OpenRouteService API through the instantiation of the OpenRouteService client. This is the OpenRouteService [API documentation](https://openrouteservice.org/dev/#/api-docs/introduction) for ORS Core-Version 9.0.0. 

Generate a [API Key](https://openrouteservice.org/dev/#/home?tab=1) (Token) it is necessary to sign up at the OpenRouteService dashboard by using your E-mail address or sign up with your GitHub. After logging in, go to the Dashboard by clicking on your profile icon and navigate to the API Keys section. Click "Create API Key" to generate a free key and then choose a service plan (the free plan has limited requests per day). Copy the API Key and store it securely. 

OpenRouteService primarily uses API keys for authentication. However, if a token is required for certain endpoints, you can send a request with your API key in the Authorization header. This process facilitated various geospatial analysis functions, including isochrone generation.


### Option 1: Using an ORS API Key
Make sure you have a .env file in the root directory with the following content:
```bash
    OPENROUTESERVICE_API_KEY='your_api_key'
```

In [ ]:
# Read the api key from the .env file
%load_ext dotenv
%dotenv
api_key = os.getenv('OPENROUTESERVICE_API_KEY')
client = openrouteservice.Client(key=api_key)

### Setting up relevant processing folders

There are different data sources used across the notebook. To handle these data sets, it is recommended to use three directories for input, temp and output data. Some of the files are related to healthcare facilities, population data. The healthcare facilities data is usualy the result of gathering global or national datasets and then carrying out local validation according to the local context. 

Despite being official, administrative boundaries may not reflect the actual patterns of human settlement or economic activity. Therefore, the team used the Functional Urban Area (FUA) as a complementary definition of the study areas. The FUA is defined by [the Joint Research Centre of the European Commission](https://commission.europa.eu/about/departments-and-executive-agencies/joint-research-centre_en) as the actual urban sprawl and human activities, encompassing the core city and economically or socially integrated surrounding regions. The FUA was obtained from [the Global Human Settlement Layer (GHSL) ](https://human-settlement.emergency.copernicus.eu/)dataset, which provides spatial data for functional urban areas worldwide. 

The following datasets are considered as input data for the analysis:


* [Datasets of health facilities](../scripts/Kano/data-inputs/healthcare_facilities.geojson)
* [Population: Women in childbearing age](../scripts/Kano/data-inputs/kano_nga_f_15_49_2015_1km.tif) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447)
* [Study Area](../../../docs/study-areas/grid-boundary-kano.gpkg) defined by the IDEAMAPS team

In [3]:
# Set paths to access Kano data
# Define directories
data_inputs = '../Data/raw/'
data_temp = '../Data/processed/'
model_outputs = '../Data/outputs/'

## Spatial Analysis Pipeline

### Travel time and dista calculation using OpenRouteService (ORS)

Using OpenRouteService (ORS) Matrix API to calculate the travel time and distance from each population grid centroid to the healthcare facility. There are two options to process the time and distance calculations: Using the public ORS API or using a local instance of the ORS server.

note: this will generate a file 'OD_matrix_healthcare_pop_grid‘

In [ ]:
# 1. Load the origin and destination datasets as GeoDataFrames
origin_gdf = gpd.read_file(data_temp + 'pop-grid-kano-centroids.gpkg')
origin_name_column = 'grid_code'

healthcare_facilities = gpd.read_file(data_inputs + 'healthcare-facilities.gpkg')
destination_gdf = healthcare_facilities.dropna(subset=['geometry'])
destination_name_column = 'facility_name'

origins = list(zip(origin_gdf.geometry.x, origin_gdf.geometry.y))
destinations = list(zip(destination_gdf.geometry.x, destination_gdf.geometry.y))
locations = origins + destinations

In [ ]:
# 2. Generate the OD-Matrix using the OpenRouteService API
origins_index = list(range(0, len(origins)))
destinations_index = list(range(len(origins), len(locations)))

body = {'locations': locations,
       'destinations': destinations_index,
       'sources': origins_index,
       'metrics': ['distance', 'duration']}

headers = {
    'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
    'Authorization': api_key,
    'Content-Type': 'application/json; charset=utf-8'
}

response = requests.post('https://api.openrouteservice.org/v2/matrix/driving-car', json=body, headers=headers)

# Extract the distance and duration matrices from the response
distances = response.json().get('distances', [])
durations = response.json().get('durations', [])

In [ ]:
# 3. Create a combined matrix of distances and durations for each origin-destination pair

distances_duration_matrix = []

# Iterate over each origin (grid)
for origin_index, origin in origin_gdf.iterrows():
    origin_name = origin[origin_name_column]
    origin_x = origin.geometry.x
    origin_y = origin.geometry.y
    origin_distances = distances[origin_index]
    origin_durations = durations[origin_index]

    # find the minimum duration and the index of the minimum duration
    min_duration = min(origin_durations)
    min_index = origin_durations.index(min_duration)
    destination_index = destinations_index[min_index]
    dest_x, dest_y = locations[destination_index]
    filtered = healthcare_facilities[(healthcare_facilities.geometry.x == dest_x) & (healthcare_facilities.geometry.y == dest_y) ]
    destination_row = filtered.iloc[0]
    dest_name = destination_row[destination_name_column]

        # Append both the distance and duration for this origin-destination pair
    distances_duration_matrix.append([
            origin_name, origin_y, origin_x,
            dest_name, dest_y, dest_x,
            min_duration
        ])

In [ ]:
# 4. Convert the combined matrix into a DataFrame and save it as a CSV file
matrix_df = pd.DataFrame(distances_duration_matrix, columns=[
    'grid_code','origin_lat', 'origin_lon',
    'destination_name', 'dest_lat', 'dest_lon','min_duration'
])

merged_df = pd.merge(matrix_df, origin_gdf[['grid_code', 'population']], on='grid_code', how='left')
merged_df.to_csv(data_temp + 'distance_duration_matrix_temp.csv', index=False)

In [ ]:
# 5. Convert the DataFrame to a GeoDataFrame and save it as a GeoPackage
geometry = [Point(xy) for xy in zip(merged_df['dest_lon'], merged_df['dest_lat'])]
gdf = gpd.GeoDataFrame(merged_df, geometry=geometry, crs="EPSG:4326")

gpkg_path = data_temp + 'distance_duration_matrix_temp.gpkg'
gdf.to_file(gpkg_path, layer="duration_matrix", driver="GPKG")

### Option 2: Using a local ORS service
Make sure you have set a local service that runs the OSM-based ORS API. 
```r
# Insert R code from the local ORS service
```

### Procedure for Computing the OD Matrix Using a Local Docker Environment

This section outlines the steps required to compute the Origin-Destination (OD) matrix using a local Docker environment. 

1. **Set Up Docker Environment**:

2. **Prepare Input Data**:

3. **Run the OD Matrix Computation Script**:

4. **Monitor the Process**:

5. **Retrieve and Validate Output**:

### Diego please add description here

## Processing OD Matrix

Population data is the result of combining 1km grid data with 100m grid data. See [Section 2]() for more details.

In [13]:
# 1. Load the destination datasets as GeoDataFrames
centroids_df = gpd.read_file(data_temp +'pop-grid-kano-centroids.gpkg')
print(f"\n✓ Loaded {len(centroids_df)} grid centroids")
print(centroids_df.head())


✓ Loaded 167260 grid centroids
   rowid   latitude  longitude   lon_min    lat_min   lon_max    lat_max  \
0      1  12.122137   8.301005  8.300491  12.121729  8.301519  12.122545   
1      2  12.072376   8.319272  8.318758  12.071968  8.319786  12.072784   
2      3  12.110716   8.330126  8.329612  12.110308  8.330640  12.111124   
3      4  12.108269   8.330079  8.329565  12.107861  8.330593  12.108676   
4      5  12.027513   8.332575  8.332061  12.027105  8.333088  12.027921   

   grid_id  bcount  pop_grid_id  pop_grid_bcount  pop_weight  pop_grid_pop  \
0        0    10.0          870            110.0    0.090909    120.140793   
1        1     1.0         1221              8.0    0.125000     94.052826   
2        2    37.0          932           1030.0    0.035922    226.764771   
3        3    77.0          932           1030.0    0.074757    226.764771   
4        4    41.0         1512            658.0    0.062310    102.597412   

         pop                              

In [14]:
# 2. Load healthcare facilities data
healthcare_facilities = gpd.read_file(data_inputs +'healthcare_facilities.geojson')
print(f"\n✓ Loaded {len(healthcare_facilities)} healthcare facilities")
print(healthcare_facilities.head())


✓ Loaded 145 healthcare facilities
   orig_order  state       lga               ward  urban_conurb         uid  \
0        1210      9     Fagge           Kwachiri             9  12757068.0   
1        1208      9     Fagge          Fagge D 2             9  23158449.0   
2        1302      9   Tarauni  Gyadi-Gyadi Arewa             9  40297833.0   
3        1277      9  Nasarawa   Tudun Wada (NSR)             9  42838223.0   
4        1327      9   Tarauni        Babban Giji             9         NaN   

      facility_code  ontime_code                        facility_name  \
0  19/12/1/2/1/0004    100904010  465 Nigerian Airforce Base Hospital   
1  19/12/1/1/1/0001    100904008         Abubakar Imam Urology Centre   
2  19/21/1/1/2/0004    100911002                        Access Clinic   
3  19/31/1/2/2/0001    100910010            Ahmadiyya Muslim Hospital   
4               NaN    100911027  Ahmed Memorial Clinic and Maternity   

   reg_number  ... longitude operation_status  reg

In [ ]:
# 3. Load the OD-Matrix CSV file into a DataFrame
matrix_df = pd.read_csv(data_temp +'OD-matrix-kano-access-emoc.csv')
print(f"\n✓ Loaded OD-Matrix with {len(matrix_df)} records")
print(matrix_df.head())


✓ Loaded OD-Matrix with 24252700 records
   origin_id  destination_id  duration_seconds  distance_km
0          1               1           1843.72        30.35
1          1               2           1724.08        27.87
2          1               3           1573.44        25.06
3          1               4           1510.41        24.80
4          1               5           2269.93        28.70


**GRID CELLS WITHOUT TRAVEL TIME ESTIMATE**

If a grid cell has a NULL value in the travel estimate, we will remove it from the analysis. This is because we cannot calculate the E2SFCA without a travel time estimate.

In [4]:
# Removing rows with NaN values in the 'duration_seconds' column
matrix_df = matrix_df.dropna(subset=['duration_seconds'])
print(f"\n✓OD-Matrix, remaining records: {len(matrix_df)}")


✓OD-Matrix, remaining records: 24249655


To process the OD Matrix we need merge it to create an integrated dataset that combines data from the healthcare facilities and population grid.For doing so, we will use the pandas library and join functions based on the id columns of all datasets.

In [ ]:
# 4. Merge the OD-Matrix DataFrame with the centroids GeoDataFrame to create a combined dataset for analysis
pop_centroids_hcf = pd.merge(matrix_df, centroids_df[['rowid', 'longitude', 'latitude', 'lon_min', 'lat_min', 'lon_max', 'lat_max','bcount','pop_grid_bcount', 'pop_grid_pop', 'pop', 'geometry']], 
                     left_on='destination_id', right_on='rowid', how='left')
print(f"\n✓ Merged OD-Matrix with centroids, resulting in {len(pop_centroids_hcf)} records")
print(pop_centroids_hcf.head())


✓ Merged OD-Matrix with centroids, resulting in 24249655 records
   origin_id  destination_id  duration_seconds  distance_km  rowid  longitude  \
0          1               1           1843.72        30.35      1   8.301005   
1          1               2           1724.08        27.87      2   8.319272   
2          1               3           1573.44        25.06      3   8.330126   
3          1               4           1510.41        24.80      4   8.330079   
4          1               5           2269.93        28.70      5   8.332575   

    latitude   lon_min    lat_min   lon_max    lat_max  bcount  \
0  12.122137  8.300491  12.121729  8.301519  12.122545    10.0   
1  12.072376  8.318758  12.071968  8.319786  12.072784     1.0   
2  12.110716  8.329612  12.110308  8.330640  12.111124    37.0   
3  12.108269  8.329565  12.107861  8.330593  12.108676    77.0   
4  12.027513  8.332061  12.027105  8.333088  12.027921    41.0   

   pop_grid_bcount  pop_grid_pop        pop  \
0  

In [8]:
pop_centroids_hcf = pop_centroids_hcf.rename(columns={
    "longitude": "origin_lon",
    "latitude": "origin_lat",
    "lon_min": "origin_lon_min",
    "lat_min": "origin_lat_min",
    "lon_max": "origin_lon_max",
    "lat_max": "origin_lat_max",
    "rowid": "grid_id",
    "origin_id": "hcf_uid",
    "pop": "population"
})
columns_to_keep = ["grid_id", "origin_lon", "origin_lat", "origin_lon_min","origin_lat_min","origin_lon_max","origin_lat_max","population", "bcount","pop_grid_bcount", "pop_grid_pop","geometry", "hcf_uid", "duration_seconds", "distance_km"]
pop_centroids_hcf = pop_centroids_hcf[columns_to_keep]

Merging the dataframe than contains the od matrix (with the healthcare facility class) and the population data with the full information about health care facilities.

In [15]:
# 5. Merge the combined dataset with the healthcare facilities dataset to include facility details and validation status
distances_duration_matrix = pd.merge(pop_centroids_hcf, healthcare_facilities[['hcf_id','facility_name', 'longitude', 'latitude', 'Local_Validation']], 
                     left_on='hcf_uid', right_on='hcf_id', how='left')

distances_duration_matrix = distances_duration_matrix.rename(columns={
    "longitude": "dest_lon",
    "latitude": "dest_lat"
})
distances_duration_matrix = distances_duration_matrix.drop(columns=['hcf_uid'])

category_counts = healthcare_facilities['Local_Validation'].value_counts()
print(category_counts)

Local_Validation
Private Comprehensive EmOC                                 105
Public Comprehensive EmOC                                   17
No EmOC                                                      9
Public/Private Basic EmOC                                    5
Private Basic EmOC                                           5
Non Functional centre .                                      2
Public Basic EmOC                                            1
Public/Private comprehensive EmOC (missionary Hospital)      1
Name: count, dtype: int64


In [ ]:
# 6. Standardize the 'Local_Validation' categories to ensure consistency in the analysis
distances_duration_matrix['Local_Validation'] = distances_duration_matrix['Local_Validation'].replace({
    'Public/Private Basic EmOC': 'Private Basic EmOC',
    'Public/Private comprehensive EmOC (missionary Hospital)': 'Private Comprehensive EmOC'
})

selected_categories = ['Public Comprehensive EmOC', 'Private Comprehensive EmOC', 
                       'Private Basic EmOC', 'Public Basic EmOC']

distances_duration_matrix = distances_duration_matrix[
    distances_duration_matrix['Local_Validation'].isin(selected_categories)]

In [ ]:
# Step 1: Create subsets based on categories of 'Validation of HCFs Categorization'
categories = {
    "public_comprehensive_EmOC": ["Public Comprehensive EmOC"],
    "private_comprehensive_EmOC": ["Private Comprehensive EmOC"],
    "private_basic_EmOC": ["Private Basic EmOC"],
    "public_basic_EmOC": ["Public Basic EmOC"]
}

subsets = {
    key: distances_duration_matrix[
        distances_duration_matrix['Local_Validation'].str.contains('|'.join(values), na=False)
    ]
    for key, values in categories.items()
}

public_CEmOC = subsets["public_comprehensive_EmOC"]
private_CEmOC = subsets["private_comprehensive_EmOC"]
public_BEmOC = subsets["public_basic_EmOC"]
private_BEmOC = subsets["private_basic_EmOC"]

In [75]:
# Step 2: Define a function to get 3 smallest duration_seconds per grid_id for each category
def get_closest_3(df, n=3):
    return (df.sort_values(['grid_id', 'duration_seconds'])
              .groupby('grid_id', as_index=False)
              .head(n)
              .reset_index(drop=True))

In [76]:
# Step 3: If the subsets are already created for each category, we apply the function to each subset:
public_CEmOC_closest_3 = get_closest_3(public_CEmOC)
private_CEmOC_closest_3 = get_closest_3(private_CEmOC)
public_BEmOC_closest_3 = get_closest_3(public_BEmOC)
private_BEmOC_closest_3 = get_closest_3(private_BEmOC)

In [ ]:
# Step 4: Concatenate the filtered results into a single DataFrame
distances_duration_matrix = pd.concat([
    public_CEmOC_closest_3, private_CEmOC_closest_3,
    public_BEmOC_closest_3, private_BEmOC_closest_3
])
distances_duration_matrix.head()

In [ ]:
# 5. Convert the final DataFrame to a GeoDataFrame and save it as a GeoPackage
# geometry = [Point(xy) for xy in zip(distances_duration_matrix['origin_lon'], distances_duration_matrix['origin_lat'])]
distances_duration_matrix = gpd.GeoDataFrame(distances_duration_matrix, geometry=geometry, crs="EPSG:4326")

gpkg_path = data_temp + 'distances_duration_3_closet_Emoc.gpkg'
distances_duration_matrix.to_file(gpkg_path, driver="GPKG")

## Enhanced Two-Step Floating Catchment Area (E2SFCA) method

In [17]:
# Function
from math import *
d = 10 * 60 # try max duration 5/10mins/15mins/20 car, under estimation of travel time and traffic condition realted to the selected data sourse 
W = 0.01
beta = - d ** 2 / log(W)
print(beta)

78173.00674258533


In [16]:
origin_dest = gpd.read_file(data_temp + 'distances_duration_3_closet_Emoc.gpkg')
print(origin_dest.head())

   grid_id  origin_lon  origin_lat  origin_lon_min  origin_lat_min  \
0        1    8.301005   12.122137        8.300491       12.121729   
1        1    8.301005   12.122137        8.300491       12.121729   
2        1    8.301005   12.122137        8.300491       12.121729   
3        2    8.319272   12.072376        8.318758       12.071968   
4        2    8.319272   12.072376        8.318758       12.071968   

   origin_lon_max  origin_lat_max  population  bcount  pop_grid_bcount  \
0        8.301519       12.122545   10.921890    10.0            110.0   
1        8.301519       12.122545   10.921890    10.0            110.0   
2        8.301519       12.122545   10.921890    10.0            110.0   
3        8.319786       12.072784   11.756603     1.0              8.0   
4        8.319786       12.072784   11.756603     1.0              8.0   

   pop_grid_pop  duration_seconds  distance_km  hcf_id  \
0    120.140793            374.30         5.77      14   
1    120.140793   

In [18]:
# 1. Convert 'duration' to numeric, coercing errors to NaN
origin_dest = origin_dest.copy()
origin_dest['duration_seconds'] = pd.to_numeric(origin_dest['duration_seconds'], errors='coerce')

In [19]:
# 2. Drop rows with NaN values in 'duration' column
origin_dest = origin_dest.dropna(subset=['duration_seconds'])
origin_dest['grid_id'] = pd.to_numeric(origin_dest['grid_id'], errors='coerce')
origin_dest_acc = origin_dest  # Backup

In [20]:
# 3. Apply Gaussian decay function to calculate the weight of each grid to healthcare 
# facilities based on the travel duration. d is the travel time and beta is the decay 
# parameter previously calculated.
# The weight decreases as the duration increases, meaning facilities that are further away have less impact.
origin_dest_acc['Weight'] = origin_dest_acc['duration_seconds'].apply(lambda d: round(math.exp(-d**2/beta), 8))

In [21]:
# Compute the Weighted Population (Pop_W), the population of each grid cell is multiplied 
# by the corresponding weight to calculate the weighted population.
origin_dest_acc['Pop_W'] = origin_dest_acc['population'] * origin_dest_acc['Weight']
print(len(origin_dest_acc))
origin_dest_acc.head()

1672390


,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,duration_seconds,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation,geometry,Weight,Pop_W
0,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,374.30,5.77,14,Dawakin Tofa General Hospital,8.331265,12.107341,Public Comprehensive EmOC,POINT (8.30101 12.12214),0.166596,1.819541
1,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,1679.81,27.83,3,Mariya Sanusi General Hospital,8.473443,12.056152,Public Comprehensive EmOC,POINT (8.30101 12.12214),0.000000,0.000000
2,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,1708.62,28.22,145,Waziri Shehu Gidado General Hospital,8.471150,12.058087,Public Comprehensive EmOC,POINT (8.30101 12.12214),0.000000,0.000000
3,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,470.08,5.20,14,Dawakin Tofa General Hospital,8.331265,12.107341,Public Comprehensive EmOC,POINT (8.31927 12.07238),0.059205,0.696052
4,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,1560.17,25.35,3,Mariya Sanusi General Hospital,8.473443,12.056152,Public Comprehensive EmOC,POINT (8.31927 12.07238),0.000000,0.000000


In [24]:
# 4. Sum the weighted population for each grid cell to get the total weighted population (Pop_W) for each grid cell.
origin_dest_sum = origin_dest_acc.groupby(by='hcf_id')['Pop_W'].sum().reset_index()
origin_dest_sum

,hcf_id,Pop_W
0,1,15603.539814
1,2,36512.813787
2,3,10659.897476
3,4,16032.091170
4,5,55939.302647
...,...,...
129,141,1666.569144
130,142,2708.502060
131,143,5616.732398
132,144,23939.736976


In [25]:
# 5. Merge the total weighted population back to the original DataFrame to have a complete dataset for analysis
origin_dest_acc = origin_dest_acc.merge(origin_dest_sum, on='hcf_id')
origin_dest_acc.head()

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation,geometry,Weight,Pop_W_x,Pop_W_y
0,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,5.77,14,Dawakin Tofa General Hospital,8.331265,12.107341,Public Comprehensive EmOC,POINT (8.30101 12.12214),0.166596,1.819541,4406.686022
1,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,27.83,3,Mariya Sanusi General Hospital,8.473443,12.056152,Public Comprehensive EmOC,POINT (8.30101 12.12214),0.000000,0.000000,10659.897476
2,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,28.22,145,Waziri Shehu Gidado General Hospital,8.471150,12.058087,Public Comprehensive EmOC,POINT (8.30101 12.12214),0.000000,0.000000,8555.357581
3,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,5.20,14,Dawakin Tofa General Hospital,8.331265,12.107341,Public Comprehensive EmOC,POINT (8.31927 12.07238),0.059205,0.696052,4406.686022
4,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,25.35,3,Mariya Sanusi General Hospital,8.473443,12.056152,Public Comprehensive EmOC,POINT (8.31927 12.07238),0.000000,0.000000,10659.897476


In [26]:
# supply value is set to 1 for simplicity (capacity of HCF)
# supply = 1
# in the future, we will link supply with ownership and EmOC service level
origin_dest_acc = origin_dest_acc.rename(columns={'Pop_W_y': 'Pop_W_S'})  # Pop_W_S: Population Weight Sum

In [27]:
# The supply value is set based on the type of healthcare facility, with different weights assigned to public and private facilities, as well as comprehensive and basic EmOC services. This reflects the varying levels of accessibility and quality of care provided by different types of healthcare facilities.
supply_map = {
    'Public Comprehensive EmOC': 1,
    'Private Comprehensive EmOC': 0.7,
    'Public Basic EmOC': 0.5,
    'Private Basic EmOC': 0.35
}

In [29]:
# Calculate the supply-demand ratio by dividing the supply value by the total weighted population (Pop_W_S) for each grid cell. This ratio provides an indication of the accessibility of healthcare facilities relative to the demand from the population in each grid cell.
origin_dest_acc['supply'] = origin_dest_acc['Local_Validation'].map(supply_map)
origin_dest_acc['supply_demand_ratio'] = origin_dest_acc['supply'] / origin_dest_acc['Pop_W_S']
origin_dest_acc['supply_demand_ratio'] = (
    origin_dest_acc['supply_demand_ratio']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

In [30]:
# Calculate Rj * Weight for Each Grid Cell
origin_dest_acc['supply_W'] = origin_dest_acc['supply_demand_ratio'] * origin_dest_acc.Weight

In [31]:
# Compute Accessibility Index (Ai) for Each Grid Cell
origin_dest_acc['Accessibility'] = origin_dest_acc.groupby('grid_id')['supply_W'].transform('sum')

In [32]:
# Normalize the Accessibility Index using Min-Max Scaling to bring the values between 0 and 1, making it easier to interpret and compare across different grid cells.
scaler = MinMaxScaler()
origin_dest_acc['Accessibility_standard'] = scaler.fit_transform(origin_dest_acc[['Accessibility']])
origin_dest_acc.head()

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,Local_Validation,geometry,Weight,Pop_W_x,Pop_W_S,supply,supply_demand_ratio,supply_W,Accessibility,Accessibility_standard
0,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,Public Comprehensive EmOC,POINT (8.30101 12.12214),0.166596,1.819541,4406.686022,1.0,0.000227,0.000038,0.000038,0.004124
1,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,Public Comprehensive EmOC,POINT (8.30101 12.12214),0.000000,0.000000,10659.897476,1.0,0.000094,0.000000,0.000038,0.004124
2,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,Public Comprehensive EmOC,POINT (8.30101 12.12214),0.000000,0.000000,8555.357581,1.0,0.000117,0.000000,0.000038,0.004124
3,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,Public Comprehensive EmOC,POINT (8.31927 12.07238),0.059205,0.696052,4406.686022,1.0,0.000227,0.000013,0.000013,0.001466
4,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,Public Comprehensive EmOC,POINT (8.31927 12.07238),0.000000,0.000000,10659.897476,1.0,0.000094,0.000000,0.000013,0.001466


In [ ]:
# Check the maximum value of the standardized Accessibility Index to ensure it is correctly normalized to 1.
max(origin_dest_acc.Accessibility_standard)

1.0

In [34]:
# Convert the final DataFrame to a GeoDataFrame and save it as a GeoPackage
gdf = gpd.GeoDataFrame(origin_dest_acc, geometry='geometry', crs="EPSG:4326")
gpkg_path = data_temp + 'acc_score_3closest.gpkg'
gdf.to_file(gpkg_path, layer="acc_score_3closest", driver="GPKG")

# 4. Grouping by grid ID to prepare the final output file
There is a need to update this part of the code

In [35]:
# Read the GeoPackage file (if starting from this section)
results_grid = gpd.read_file(data_temp + 'acc_score_3closest.gpkg')

# Select columns to keep and reorder them
results_grid = results_grid[['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 
                             'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'hcf_id', 
                             'facility_name', 'Local_Validation', 'duration_seconds', 'distance_km', 
                             'Accessibility_standard', 'geometry']]

In [36]:
# For each grid cell, keep the row with the minimum duration_seconds (closest healthcare facility)
idx = results_grid.groupby('grid_id')['duration_seconds'].idxmin()
results_grid_dedup = results_grid.loc[idx].reset_index(drop=True)
print(len(results_grid_dedup))
results_grid_dedup.head()

167239


,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,hcf_id,facility_name,Local_Validation,duration_seconds,distance_km,Accessibility_standard,geometry
0,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,374.30,5.77,0.004124,POINT (8.30101 12.12214)
1,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,470.08,5.20,0.001466,POINT (8.31927 12.07238)
2,3,8.330126,12.110716,8.329612,12.110308,8.330640,12.111124,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,104.01,0.48,0.021559,POINT (8.33013 12.11072)
3,4,8.330079,12.108269,8.329565,12.107861,8.330593,12.108676,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,40.99,0.22,0.024239,POINT (8.33008 12.10827)
4,5,8.332575,12.027513,8.332061,12.027105,8.333088,12.027921,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,1518.11,12.65,0.000000,POINT (8.33257 12.02751)


In [37]:
print(results_grid_dedup['Local_Validation'].value_counts())

Local_Validation
Private Comprehensive EmOC    120825
Public Comprehensive EmOC      36954
Private Basic EmOC              9203
Public Basic EmOC                257
Name: count, dtype: int64


In [38]:
# Save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'EmOC-deprivation-access.gpkg'
results_grid_dedup.to_file(output_gpkg_path, layer='EmOC-deprivation-access', driver='GPKG')

### Setting values for Low medium and High categories

We started by defining equal value division, and modified the thesholds to a value that is more legible and easier to interpret. Every model should have their own thresholds based on the data distribution of the three categories. 

Note: For Kano, we excluded grid cells with index values below 0.000001 that indicated very low population and a small number of buildings.  

In [5]:
# Classify the accessibility into three categories based on the standardized Accessibility Index: 0 for High accessibility, 1 for Medium accessibility, and 2 for Low accessibility. The thresholds for classification are set at 0.005 and 0.02, which can be adjusted based on the distribution of the Accessibility Index in the dataset.
results_grid_dedup['result'] = 2  # Initialize all cells to 2 (Low accessibility)
results_grid_dedup.loc[results_grid_dedup['Accessibility_standard'] > 0.005, 'result'] = 1
results_grid_dedup.loc[results_grid_dedup['Accessibility_standard'] > 0.02, 'result'] = 0

In [6]:
category_counts = results_grid_dedup['result'].value_counts()
print(category_counts)

result
2    130081
1     24886
0     12272
Name: count, dtype: int64


### Setting values for focus areas

We defined the focus areas based on values for the different thresholds. We aim at participants helping us to confirm the selection of the city-specific thresholds.

In [7]:
# Identify focus areas for intervention based on the standardized Accessibility Index.
results_grid_dedup['focused'] = 0
# Focus areas between the Low category and the excluded cells due to low population or no buildings
results_grid_dedup.loc[(results_grid_dedup['Accessibility_standard'] > 0.000001) & (results_grid_dedup['Accessibility_standard'] < 0.0000015), 'focused'] = 1
# Focus areas between the Medium and High categories
results_grid_dedup.loc[(results_grid_dedup['Accessibility_standard'] > 0.003) & (results_grid_dedup['Accessibility_standard'] < 0.006), 'focused'] = 1
# Focus areas between the Low and Medium categories
results_grid_dedup.loc[(results_grid_dedup['Accessibility_standard'] > 0.019) & (results_grid_dedup['Accessibility_standard'] < 0.03), 'focused'] = 1

In [8]:
category_counts = results_grid_dedup['focused'].value_counts()
print(category_counts)

focused
0    147064
1     20175
Name: count, dtype: int64


In [9]:
# Rename columns for clarity and consistency
results_grid_dedup = results_grid_dedup.rename(columns={
    'origin_lon': 'longitude',
    'origin_lat': 'latitude',
    'origin_lon_min': 'lon_min',
    'origin_lat_min': 'lat_min',
    'origin_lon_max': 'lon_max',
    'origin_lat_max': 'lat_max',
    'hcf_id': 'closest_hcf_id',
    'facility_name': 'closest_facility_name',
    'duration_seconds': 'closest_duration(seconds)',
    'distance_km': 'closest_distance(km)',
    'Accessibility_standard': 'Accessibility_Index_Standard'
})

results_grid_dedup.head()

,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,closest_hcf_id,closest_facility_name,Local_Validation,closest_duration(seconds),closest_distance(km),Accessibility_Index_Standard,geometry,result,focused
0,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,374.30,5.77,0.004124,POINT (8.30101 12.12214),2,1
1,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,470.08,5.20,0.001466,POINT (8.31927 12.07238),2,0
2,3,8.330126,12.110716,8.329612,12.110308,8.330640,12.111124,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,104.01,0.48,0.021559,POINT (8.33013 12.11072),0,1
3,4,8.330079,12.108269,8.329565,12.107861,8.330593,12.108676,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,40.99,0.22,0.024239,POINT (8.33008 12.10827),0,1
4,5,8.332575,12.027513,8.332061,12.027105,8.333088,12.027921,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,1518.11,12.65,0.000000,POINT (8.33257 12.02751),2,0


In [10]:
# Dataset 1:
dataset_1 = results_grid_dedup.drop(columns=['geometry'])
output_gpkg_path = model_outputs + 'deprivation_classification.geojson'
dataset_1.to_file(output_gpkg_path, layer='deprivation_classification.geojson', driver='GeoJSON')

AttributeError: 'DataFrame' object has no attribute 'to_file'

In [ ]:
# Dataset 2: 
dataset_2 = results_grid_dedup.drop(columns=['grid_id', 'closest_hcf_id', 'closest_facility_name', 
                                             'Local_Validation', 'closest_duration(seconds)', 
                                             'closest_distance(km)', 'geometry'])
output_gpkg_path = model_outputs + 'accessibility-index-class.geojson'
dataset_2.to_file(output_gpkg_path, layer='accessibility-index-class.geojson', driver='GeoJSON')

In [ ]:
# Dataset 3:
dataset_3 = healthcare_facilities
output_gpkg_path = model_outputs + 'healthcare_facilities.geojson'
dataset_3.to_file(output_gpkg_path, layer='healthcare-facilities.geojson', driver='GeoJSON')

In [64]:
# Summarize the results by duration and distance for each category of accessibility (Low, Medium, High)
results_grid_dedup['duration_minutes'] = results_grid_dedup['duration_seconds'] / 60

summary = results_grid_dedup.groupby(['Local_Validation', 'result']).agg({
    'duration_minutes': 'mean',
    'distance_km': 'mean'
}).round(2)

summary = summary.rename(columns={
    'duration_minutes': 'Avg_Duration_Min',
    'distance_km': 'Avg_Distance_KM'
})

summary.index = summary.index.set_levels(
    summary.index.levels[1].map({0: 'Low', 1: 'Medium', 2: 'High'}),
    level=1
)

print(summary)

                                   Avg_Duration_Min  Avg_Distance_KM
Local_Validation           result                                   
Private Basic EmOC         Low                 1.56             0.78
                           Medium              2.75             1.39
                           High               11.59             9.85
Private Comprehensive EmOC Low                 2.21             1.31
                           Medium              3.96             2.66
                           High               12.85            10.66
Public Basic EmOC          Low                 2.07             1.08
                           Medium              3.49             1.71
Public Comprehensive EmOC  Low                 2.96             2.26
                           Medium              4.66             3.48
                           High               13.62            10.70
